# D4-3: the varying-goal check (validation-only; the test family is never generated)

Follow-up to D4-2 (`docs/research-notes/2026-09-30-d4-2-flop-priced-scoring.md`, hypothesis 2). The full design, the prices, the frozen rules and the list of changes made before this run are in `docs/plans/d4-3-plan.md`; read that first. **This notebook is a design study, not a confirmatory gate.** It uses the D4-2 tuning (seed 3003) and validation (seed 1001) instances of the seven D4-2 cells with **one change: each instance gets its own random unit goal** (`highdim.varying_goal_instances`). **The test family (seed 2002) is never generated or read.** It runs on CPU and trains nothing.

**What it does.** For each cell it compares, at equal compute: `uniform_pass`; the step-doubling arms `residual` (goal-blind), `goal_local` and `adjoint` (scoring charged `2n`, `2n`, `3n` CN steps per pass); the cheap, solve-free arms `cheap` (goal-blind) and `cheap_adjoint` (weighted by the tabulated co-state of the instance's own goal, **table taken as given**); `cheap_adjoint_setup` (the same traces with the table's per-instance setup charged); and the hypothetical ceiling `adjoint_free` (the `adjoint` arm with scoring priced at zero). `θ` is chosen per (cell, arm) on the tuning family; compute to reach 10 %, 3 % and 1 % of the initial-to-finest gap is priced by log-log interpolation (primary), with semilog and staircase as sensitivities.

**Frozen rules** (plan §6; "beats" is the CI of the mean log compute ratio below 0 at two adjacent targets):

- **R3v, the co-state weight helps a cheap estimator:** `cheap_adjoint` beats `cheap`.
- **R4v, adaptivity pays with a goal-aware cheap arm:** `cheap_adjoint` beats `uniform_pass`.
- **D4-1 candidate cell:** R3v and R4v both hold.

**Order.** Tuning instances are generated and checked, `θ` is chosen on them and everything is frozen and hashed; only then are the validation instances generated and run.

**Limits.** No learned critic, so no H2 statement. The goal is a random unit vector; `m` is confounded with the system draw; `base`, `depth7` and `depth11` are near copies of one configuration. The instances are generated from frozen seeds and are **not real data**.

In [ ]:
# ================================================================
# 0. Package (CPU is enough)
# ================================================================
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/romusking/para_001'
REPO_REF = 'main'  # branch, tag or commit SHA
REPO_DIR = '/content/para_001' if IN_COLAB else str(Path.cwd().resolve().parents[1])

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', '--quiet', REPO_URL, REPO_DIR], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--quiet', '--all'], check=False)
    for candidate in (f'origin/{REPO_REF}', REPO_REF):
        resolved = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', '--verify', '--quiet', candidate + '^{commit}'],
                                  capture_output=True, text=True)
        if resolved.returncode == 0:
            subprocess.run(['git', '-C', REPO_DIR, 'checkout', '--quiet', '--detach', resolved.stdout.strip()], check=True)
            break
    else:
        raise RuntimeError(f'Cannot resolve {REPO_REF!r}.')
REPO_COMMIT = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], text=True).strip()
REPO_DIRTY = bool(subprocess.check_output(['git', '-C', REPO_DIR, 'status', '--porcelain'], text=True).strip())
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
try:
    from adjointrwm.domains import highdim
    highdim.varying_goal_instances  # noqa: B018
except (ImportError, AttributeError) as error:
    raise RuntimeError(f'{REPO_REF!r} predates the D4-3 additions; set REPO_REF to a commit that has them.') from error

if IN_COLAB:
    from google.colab import drive

    drive.mount('/content/drive', force_remount=False)
    RUN_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/AdjointRWM_Production/runs')
else:
    RUN_ROOT = Path.home() / 'adjointrwm_runs'
print('adjointrwm at', REPO_COMMIT, '(uncommitted changes!)' if REPO_DIRTY else '', '| runs ->', RUN_ROOT)

In [ ]:
# ================================================================
# 1. Frozen configuration, the immutable run directory and the tuning instances
# ================================================================
import dataclasses
import datetime
import json
import platform
import time
from dataclasses import asdict, dataclass

import numpy as np
import pandas as pd

from adjointrwm.domains import AdaptiveTimeSteppingDomain, highdim as hd
from adjointrwm.io import atomic_write_json, atomic_write_text, sha256_json


@dataclass(frozen=True)
class D43Config:
    cells: tuple = tuple(hd.CELLS)                 # base, m4, m64, wide, amp1, depth7, depth11
    tuning_instances: int = 10                     # theta (seed 3003, goals from their own stream)
    validation_instances: int = 20                 # reported (seed 1001)
    # The test family (seed 2002) is deliberately never generated in this notebook.
    thetas: tuple = (0.5, 0.7, 0.85, 0.95, 0.98, 0.995)
    step_doubling_arms: tuple = ('residual', 'goal_local', 'adjoint')
    cheap_arms: tuple = ('cheap', 'cheap_adjoint')
    gap_fractions: tuple = (0.1, 0.03, 0.01)
    cap_in_finest_grids: float = 2.0
    max_passes: int = 40
    decision_scales: tuple = (1.0, 0.0)            # 1 = real price; 0 = hypothetical free scoring (read only for the adjoint ceiling)
    primary_method: str = 'loglog'
    methods: tuple = ('loglog', 'semilog', 'staircase')
    bootstrap_resamples: int = 10000


CFG = D43Config()
CONFIG_DICT = asdict(CFG)
CELL_TABLE = {name: asdict(hd.CELLS[name]) for name in CFG.cells}
GOAL_SEEDS = {family: hd.FAMILY_SEEDS[family] + hd.GOAL_STREAM_OFFSET for family in ('tuning', 'validation')}
CONFIG_HASH = sha256_json({'config': CONFIG_DICT, 'cells': CELL_TABLE, 'family_seeds': hd.FAMILY_SEEDS, 'goal_stream_offset': hd.GOAL_STREAM_OFFSET})
RUN_ID = 'd4_3_varying_goal_' + datetime.datetime.now(datetime.UTC).strftime('%Y%m%dT%H%M%SZ')
RUN_DIR = RUN_ROOT / RUN_ID
if RUN_DIR.exists():
    raise RuntimeError(f'{RUN_DIR} exists; run directories are immutable.')
PATHS = {k: RUN_DIR / k for k in ('config', 'artifacts', 'figures', 'reports', 'tests')}
for path in PATHS.values():
    path.mkdir(parents=True, exist_ok=True)

DOMAIN = AdaptiveTimeSteppingDomain(objective_kind='bound')
TUNING_INSTANCES = {cell: hd.varying_goal_instances(cell, 'tuning', CFG.tuning_instances) for cell in CFG.cells}


def instance_table(instances_by_cell, family):
    return pd.DataFrame([{'cell': cell, 'family': family, 'name': i.name, 'goal': list(i.goal), 'y0': list(i.y0), 'n_fine': i.n_fine,
                          'pulses_json': json.dumps([asdict(p) for p in i.pulses], separators=(',', ':'))}
                         for cell, insts in instances_by_cell.items() for i in insts])


def manifest_of(instances_by_cell, family, count):
    return {cell: {'goal_seed': GOAL_SEEDS[family], 'family_seed': hd.FAMILY_SEEDS[family], 'count': count,
                   'names': [insts[0].name, insts[-1].name], 'sha256': sha256_json([asdict(i) for i in insts])}
            for cell, insts in instances_by_cell.items()}


instance_table(TUNING_INSTANCES, 'tuning').to_parquet(PATHS['config'] / 'instances_tuning.parquet', index=False)
TUNING_MANIFEST = manifest_of(TUNING_INSTANCES, 'tuning', CFG.tuning_instances)
atomic_write_json(PATHS['config'] / 'instances_manifest_tuning.json', {**TUNING_MANIFEST, 'sha256': sha256_json(TUNING_MANIFEST)})
atomic_write_json(PATHS['config'] / 'run_config.json', {
    'run_id': RUN_ID, 'config': CONFIG_DICT, 'cells': CELL_TABLE, 'family_seeds': hd.FAMILY_SEEDS, 'goal_stream_offset': hd.GOAL_STREAM_OFFSET,
    'config_sha256': CONFIG_HASH, 'repo_commit': REPO_COMMIT, 'repo_dirty': REPO_DIRTY, 'notebook': 'notebooks/04-domains/d4_3_varying_goal.ipynb',
    'plan': 'docs/plans/d4-3-plan.md', 'test_family_read': False,
})
atomic_write_json(PATHS['config'] / 'environment.json', {
    'python': sys.version, 'platform': platform.platform(), 'numpy': np.__version__, 'pandas': pd.__version__,
    'hardware': 'CPU (FLOPs are counted, not measured)',
})
print('Run:', RUN_ID, '| config', CONFIG_HASH[:12], '|', len(CFG.cells), 'cells,', CFG.tuning_instances, 'tuning instances each')

In [ ]:
# ================================================================
# 2. Correctness and ledger checks on the tuning instances (before anything is tuned or read)
# ================================================================
from adjointrwm.domains import LinearODEInstance
from adjointrwm.domains.linear_ode import ODEState, reference_solution

rows = []
for cell in CFG.cells:
    insts = TUNING_INSTANCES[cell]
    fixed = hd.cell_instances(cell, 'tuning', CFG.tuning_instances)
    inst = insts[0]
    m, pulses = inst.m, len(inst.pulses)
    counted_step = 2 * (2 * m * m) + m + m + m + 2 * pulses * (2 * m + 8)
    nodes = tuple(range(0, inst.n_fine + 1, 2))
    state = ODEState(inst, nodes, DOMAIN.stepper(inst).solve(nodes))
    discrete = DOMAIN.costate(state)
    proxy_gap = float(np.linalg.norm(discrete - hd.costate_table(inst)[np.asarray(nodes[1:])]) / np.linalg.norm(discrete))
    other = insts[1]
    mixed = dataclasses.replace(inst, goal=tuple(0.3 * np.array(inst.goal) - 1.7 * np.array(other.goal)))
    linearity_gap = float(np.abs(hd.costate_table(mixed) - (0.3 * hd.costate_table(inst) - 1.7 * hd.costate_table(other))).max())
    goals = np.array([i.goal for i in insts])
    gram = goals @ goals.T
    start = DOMAIN.initial_state(inst)
    picked = [DOMAIN.legal_candidates(start)[j] for j in (1, 5, 6)]
    batch = DOMAIN.apply_batch(start, picked)
    weighted = DOMAIN.weighted_local_errors(batch, inst).sum()
    finer = LinearODEInstance(**{**inst.__dict__, 'max_depth': inst.max_depth + 1})
    rows.append({'cell': cell, 'm': m, 'flops_step_formula': hd.flops_step(inst), 'flops_step_counted': counted_step,
                 'costate_proxy_rel_gap_fine_mesh': proxy_gap, 'costate_linearity_max_abs_gap': linearity_gap,
                 'same_as_d4_2_except_goal_and_name': all((f.A, f.y0, f.pulses, f.max_depth) == (v.A, v.y0, v.pulses, v.max_depth) and f.goal != v.goal
                                                          for f, v in zip(fixed, insts)),
                 'goals_unit_and_distinct': bool(np.allclose(np.diag(gram), 1.0) and np.abs(gram - np.eye(len(insts))).max() < 0.999),
                 'max_abs_cosine_between_goals': float(np.abs(gram - np.eye(len(insts))).max()),
                 'representation_abs_diff': float(abs(weighted - DOMAIN.signed_error(batch, inst))), 'abs_qoi_error': abs(DOMAIN.signed_error(batch, inst)),
                 'reference_max_abs_diff_yT': float(np.abs(reference_solution(inst)[-1] - reference_solution(finer)[-1]).max()),
                 'setup_steps': hd.setup_steps(inst), 'setup_steps_price_report': hd.price_report(inst, (hd.FEATURE_DIM, 8, 8, 1))['table_setup_steps']})
checks = pd.DataFrame(rows)
probe = pd.DataFrame([(i.name, p, 'gap3%', 1.0, c) for i in TUNING_INSTANCES['m4'][:2] for p, c in (('cheap_adjoint', 50.0), ('cheap', 60.0))],
                     columns=['instance', 'policy', 'target', 'decision_scale', 'compute'])
charged = hd.with_setup_charged(probe, TUNING_INSTANCES['m4'][:2])
derived_ok = bool(all(abs(r['compute'] - (50.0 + hd.setup_steps(next(i for i in TUNING_INSTANCES['m4'] if i.name == r['instance'])))) < 1e-9
                      for _, r in charged[charged['policy'] == 'cheap_adjoint_setup'].iterrows()))
CORRECTNESS = {
    'flop_formula_matches_counted_code_path': bool((checks['flops_step_formula'] == checks['flops_step_counted']).all()),
    'costate_proxy_close_on_fine_mesh': bool((checks['costate_proxy_rel_gap_fine_mesh'] < 0.05).all()),
    'costate_table_is_linear_in_the_goal': bool((checks['costate_linearity_max_abs_gap'] < 1e-8).all()),
    'instances_equal_d4_2_except_goal_and_name': bool(checks['same_as_d4_2_except_goal_and_name'].all()),
    'goals_are_unit_and_distinct': bool(checks['goals_unit_and_distinct'].all()),
    'error_representation_on_batch_grids': bool((checks['representation_abs_diff'] <= 1e-9 * np.maximum(1.0, checks['abs_qoi_error'])).all()),
    'reference_converged': bool((checks['reference_max_abs_diff_yT'] < 1e-8).all()),
    'setup_steps_equal_the_price_report': bool(np.allclose(checks['setup_steps'], checks['setup_steps_price_report'])),
    'setup_charged_rows_equal_cheap_adjoint_plus_setup': derived_ok,
    'max_costate_proxy_rel_gap': float(checks['costate_proxy_rel_gap_fine_mesh'].max()),
    'max_abs_cosine_between_goals': float(checks['max_abs_cosine_between_goals'].max()),
    'max_representation_abs_diff': float(checks['representation_abs_diff'].max()),
    'max_reference_diff': float(checks['reference_max_abs_diff_yT'].max()),
    'test_family_refused': False,
}
try:
    hd.varying_goal_instances('base', 'test', 1)
except ValueError:
    CORRECTNESS['test_family_refused'] = True
CORRECTNESS['passed'] = bool(all(v for k, v in CORRECTNESS.items() if isinstance(v, bool)))
atomic_write_json(PATHS['tests'] / 'correctness.json', CORRECTNESS)
checks.to_csv(PATHS['tests'] / 'ledger_and_goal_checks.csv', index=False)
print(json.dumps(CORRECTNESS, indent=2))
if not CORRECTNESS['passed']:
    raise RuntimeError('Correctness failed: nothing below may be read. See tests/.')

In [ ]:
# ================================================================
# 3. Choose theta per (cell, arm) on the tuning family, then freeze
# ================================================================
from adjointrwm.domains import marking_policy, uniform_pass_policy

STEP_DOUBLING = {kind: (lambda th, kind=kind: marking_policy(kind, th)) for kind in CFG.step_doubling_arms}
CHEAP = {'cheap': hd.cheap_policy, 'cheap_adjoint': hd.cheap_adjoint_policy}
ARMS = list(CFG.step_doubling_arms) + list(CFG.cheap_arms)
SELECTED_THETA, TUNING_ROWS = {}, []
started = time.perf_counter()
for cell in CFG.cells:
    tuning = TUNING_INSTANCES[cell]
    for arm in ARMS:
        make = STEP_DOUBLING.get(arm) or CHEAP[arm]
        options = hd.tune_theta(DOMAIN, tuning, make, CFG.thetas, CFG.gap_fractions, CFG.cap_in_finest_grids, CFG.max_passes, CFG.primary_method)
        SELECTED_THETA[(cell, arm)] = min(options, key=options.get)
        TUNING_ROWS.append({'cell': cell, 'arm': arm, **{f'theta_{t:g}': options[t] for t in CFG.thetas}, 'selected': SELECTED_THETA[(cell, arm)],
                            'selected_at_grid_edge': SELECTED_THETA[(cell, arm)] in (min(CFG.thetas), max(CFG.thetas))})
    print(f'{cell}: {time.perf_counter() - started:.0f}s elapsed')
TUNING = pd.DataFrame(TUNING_ROWS)
TUNING.to_csv(PATHS['artifacts'] / 'theta_tuning.csv', index=False)
print(TUNING[['cell', 'arm', 'selected', 'selected_at_grid_edge']].to_string(index=False))

PAIRS = [('cheap', 'uniform_pass'), ('cheap_adjoint', 'uniform_pass'), ('cheap_adjoint_setup', 'uniform_pass'), ('adjoint', 'uniform_pass'),
         ('residual', 'uniform_pass'), ('goal_local', 'uniform_pass'), ('cheap_adjoint', 'cheap'), ('cheap_adjoint_setup', 'cheap'),
         ('goal_local', 'residual'), ('adjoint', 'residual'), ('adjoint_free', 'uniform_pass'), ('adjoint_free', 'cheap_adjoint')]
FROZEN = {'config_sha256': CONFIG_HASH, 'tuning_manifest_sha256': sha256_json(TUNING_MANIFEST),
          'selected_theta': {f'{c}/{a}': v for (c, a), v in SELECTED_THETA.items()}, 'pairs': [list(p) for p in PAIRS],
          'rules': {'R3v': 'cheap_adjoint beats cheap at two adjacent targets (primary method log-log)',
                    'R4v': 'cheap_adjoint beats uniform_pass at two adjacent targets',
                    'candidate_for_d4_1': 'R3v and R4v', 'beats': 'CI of mean log compute ratio below 0',
                    'cheap_adjoint_setup': 'reported, not gated'},
          'primary_method': CFG.primary_method, 'written_utc': datetime.datetime.now(datetime.UTC).isoformat()}
FROZEN['sha256'] = sha256_json({k: v for k, v in FROZEN.items() if k != 'written_utc'})
atomic_write_json(PATHS['reports'] / 'frozen_before_validation.json', FROZEN)
print('frozen', FROZEN['sha256'][:12])

In [ ]:
# ================================================================
# 4. Validation family: generated only now. Compute to reach each target at the real price
# ================================================================
from adjointrwm.domains import evaluate_work_precision, work_precision_summary

VALIDATION_INSTANCES = {cell: hd.varying_goal_instances(cell, 'validation', CFG.validation_instances) for cell in CFG.cells}
instance_table(VALIDATION_INSTANCES, 'validation').to_parquet(PATHS['config'] / 'instances_validation.parquet', index=False)
VALIDATION_MANIFEST = manifest_of(VALIDATION_INSTANCES, 'validation', CFG.validation_instances)
atomic_write_json(PATHS['config'] / 'instances_manifest_validation.json', {**VALIDATION_MANIFEST, 'sha256': sha256_json(VALIDATION_MANIFEST)})

POLICIES = {}
for cell in CFG.cells:
    arms = [uniform_pass_policy()]
    for arm in ARMS:
        make = STEP_DOUBLING.get(arm) or CHEAP[arm]
        arms.append(dataclasses.replace(make(SELECTED_THETA[(cell, arm)]), name=arm))
    POLICIES[cell] = arms
TARGETS = list(hd.TARGETS)


def cap_of(instance):
    return CFG.cap_in_finest_grids * instance.n_fine


started = time.perf_counter()
FRAMES, SUMMARY = {}, {}
for cell in CFG.cells:
    parts = [evaluate_work_precision(DOMAIN, [inst], POLICIES[cell], lambda d, i: hd.gap_targets(d, i, CFG.gap_fractions), compute_cap=cap_of(inst),
                                     max_steps=CFG.max_passes, random_draws=1, decision_scales=CFG.decision_scales, methods=CFG.methods)
             for inst in VALIDATION_INSTANCES[cell]]
    frame = pd.concat(parts, ignore_index=True)
    frame.insert(0, 'cell', cell)
    FRAMES[cell] = frame
    cap = float(cap_of(VALIDATION_INSTANCES[cell][0]))
    SUMMARY[cell] = {}
    for method in CFG.methods:
        sub = hd.with_setup_charged(hd.with_free_reference(frame[frame['method'] == method].drop(columns=['cell', 'method'])), VALIDATION_INSTANCES[cell])
        SUMMARY[cell][method] = work_precision_summary(sub, PAIRS, compute_cap=cap, num_resamples=CFG.bootstrap_resamples)['1']
    print(f'{cell}: {time.perf_counter() - started:.0f}s elapsed')
ALL = pd.concat(FRAMES.values(), ignore_index=True)
ALL.to_parquet(PATHS['artifacts'] / 'validation_work_precision.parquet', index=False)
atomic_write_json(PATHS['artifacts'] / 'validation_summary.json', {
    'selected_theta': FROZEN['selected_theta'], 'pairs': PAIRS, 'methods': list(CFG.methods), 'primary_method': CFG.primary_method, 'summary': SUMMARY,
    'note': 'adjoint_free is the adjoint arm with scoring priced at zero (hypothetical); cheap_adjoint_setup adds the per-instance table setup to cheap_adjoint; '
            'every other arm is at its real price'})
for cell in CFG.cells:
    for target in TARGETS:
        e = SUMMARY[cell][CFG.primary_method][target]
        print(f"{cell:8s} {target:6s} " + ' '.join(f"{p}={e['median_compute'][p]:.0f}" for p in ('uniform_pass', 'cheap', 'cheap_adjoint', 'cheap_adjoint_setup', 'adjoint', 'adjoint_free')))

In [ ]:
# ================================================================
# 5. Frozen rules, price ratios, table, figure and report
# ================================================================
CELL_ROWS, PRICES = [], {}
for cell in CFG.cells:
    inst = VALIDATION_INSTANCES[cell][0]
    PRICES[cell] = hd.price_report(inst, (hd.FEATURE_DIM, 8, 8, 1))
    per_method = {m: hd.cell_rules_varying_goal(SUMMARY[cell][m]) for m in CFG.methods}
    primary = per_method[CFG.primary_method]
    row = {'cell': cell, 'm': inst.m, 'R3v': primary['R3v_costate_weight_helps_cheap_estimator'], 'R4v': primary['R4v_goal_aware_cheap_arm_beats_uniform'],
           'candidate_regime_for_d4_1': primary['candidate_regime_for_d4_1'], 'setup_charged_beats_cheap': primary['setup_charged_cheap_adjoint_beats_cheap'],
           'R3v_semilog': per_method['semilog']['R3v_costate_weight_helps_cheap_estimator'], 'R3v_staircase': per_method['staircase']['R3v_costate_weight_helps_cheap_estimator'],
           'R4v_semilog': per_method['semilog']['R4v_goal_aware_cheap_arm_beats_uniform'], 'R4v_staircase': per_method['staircase']['R4v_goal_aware_cheap_arm_beats_uniform'],
           'cheap_adjoint_price_ratio_to_step_doubling': PRICES[cell]['ratio_to_step_doubling']['cheap_adjoint'],
           'table_setup_steps': PRICES[cell]['table_setup_steps']}
    for t in TARGETS:
        d = SUMMARY[cell][CFG.primary_method][t]['differences']
        row.update({f'cheap_adjoint_over_cheap_{t}': d['cheap_adjoint / cheap']['ratio_of_geometric_means'],
                    f'cheap_adjoint_over_uniform_{t}': d['cheap_adjoint / uniform_pass']['ratio_of_geometric_means'],
                    f'cheap_over_uniform_{t}': d['cheap / uniform_pass']['ratio_of_geometric_means'],
                    f'cheap_adjoint_setup_over_uniform_{t}': d['cheap_adjoint_setup / uniform_pass']['ratio_of_geometric_means'],
                    f'goal_local_over_residual_{t}': d['goal_local / residual']['ratio_of_geometric_means'],
                    f'adjoint_over_uniform_{t}': d['adjoint / uniform_pass']['ratio_of_geometric_means'],
                    f'adjoint_free_over_uniform_{t}': d['adjoint_free / uniform_pass']['ratio_of_geometric_means']})
    CELL_ROWS.append(row)
CELLS = pd.DataFrame(CELL_ROWS)
CELLS.to_csv(PATHS['figures'] / 'phase_table.csv', index=False)
print(CELLS[['cell', 'm', 'R3v', 'R4v', 'candidate_regime_for_d4_1', 'setup_charged_beats_cheap', 'R3v_semilog', 'R3v_staircase']].to_string(index=False))

import matplotlib.pyplot as plt

arms = [('cheap', 'C2'), ('cheap_adjoint', 'C3'), ('cheap_adjoint_setup', 'C5'), ('adjoint', 'C1'), ('adjoint_free', 'C4')]
fig, axes = plt.subplots(1, len(TARGETS), figsize=(4.6 * len(TARGETS), 4.2), sharey=True)
for ax, target in zip(np.atleast_1d(axes), TARGETS):
    for k, (arm, colour) in enumerate(arms):
        ax.scatter(CELLS[f'{arm}_over_uniform_{target}'], np.arange(len(CELLS)) + 0.12 * (k - 2), color=colour, label=arm, s=28)
    ax.axvline(1.0, color='grey', lw=1)
    ax.set_xscale('log'); ax.set_yticks(np.arange(len(CELLS))); ax.set_yticklabels(CELLS['cell'])
    ax.set_title(f'target: {target[3:]} of the gap left')
    ax.set_xlabel('compute vs uniform_pass (below 1: adaptive is cheaper)')
np.atleast_1d(axes)[0].legend(fontsize=8)
fig.suptitle(f'{RUN_ID}: validation, geometric-mean compute ratio (adjoint_free hypothetical; cheap_adjoint_setup charges the table per instance)', fontsize=9)
fig.savefig(PATHS['figures'] / 'compute_ratios.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

REPORT = {
    'run_id': RUN_ID, 'milestone': 'D4-3 (exploratory design study)', 'config_hash': CONFIG_HASH, 'frozen_sha256': FROZEN['sha256'], 'repo_commit': REPO_COMMIT,
    'correctness': CORRECTNESS, 'selected_theta': FROZEN['selected_theta'],
    'theta_selected_at_grid_edge': {f'{r["cell"]}/{r["arm"]}': bool(r['selected_at_grid_edge']) for r in TUNING_ROWS},
    'price_reports': PRICES, 'cells': CELLS.to_dict(orient='records'),
    'candidate_cells_for_d4_1': CELLS[CELLS['candidate_regime_for_d4_1']]['cell'].tolist(),
    'test_family_read': False,
    'claim_boundary': ('Exploratory design study on a tuning and a validation family of the analytic D4 benchmark with a random unit goal per instance; the test family was '
                       'not read. FLOPs are counted from the code path, not measured. adjoint_free is a hypothetical ceiling and cheap_adjoint takes the tabulated co-state as given. '
                       'No learned critic is involved, so no H2 statement follows.'),
}
atomic_write_json(PATHS['reports'] / 'acceptance_report.json', REPORT)
lines = [f'# D4-3 `{RUN_ID}` (exploratory)', '', f"Correctness: **{'PASS' if CORRECTNESS['passed'] else 'FAIL'}** · D4-1 candidate cells: "
         f"**{len(REPORT['candidate_cells_for_d4_1'])}** of {len(CELLS)}", '',
         '| Cell | m | R3v co-state weight helps cheap | R4v cheap-adjoint beats uniform | Candidate | Setup charged beats cheap |', '|---|---:|---|---|---|---|']
for _, r in CELLS.iterrows():
    lines.append(f"| {r['cell']} | {r['m']} | {r['R3v']} | {r['R4v']} | {r['candidate_regime_for_d4_1']} | {r['setup_charged_beats_cheap']} |")
lines += ['', REPORT['claim_boundary']]
atomic_write_text(PATHS['reports'] / 'run_summary.md', '\n'.join(lines) + '\n')
atomic_write_text(RUN_DIR / 'COMPLETE', datetime.datetime.now(datetime.UTC).isoformat() + '\n')
print('\n'.join(lines))
print('\nRun directory:', RUN_DIR)

## Reading the result

| Outcome in a cell | Meaning | Next step |
|---|---|---|
| R3v and R4v both hold | The co-state weight helps a goal-blind cheap estimator and adaptivity pays with it, averaged over goals | A candidate regime for D4-1 (a learned direct critic given the goal against a critic given co-state features); it needs its own frozen plan |
| R3v holds, R4v does not | The weight helps inside a regime where uniform refinement is still cheaper | D4-1 stays closed there |
| R4v holds, R3v does not | Goal-aware cheap scoring pays but the weight is not what pays | The co-state adds nothing to a cheap estimator there |
| Neither | The weight adds nothing and adaptivity does not pay | Record a D4 negative result for that cell |

Read the `cheap_adjoint_setup` column and the price ratios before any statement about practicality: the primary arm takes the tabulated co-state as given.